# FaceFusion trên Google Colab

**Runtime:** GPU (L4 / A100 khuyên dùng) · Python 3

**Drive:**
- Source: `/content/drive/MyDrive/input/face/` (ảnh khuôn mặt)
- Target: `/content/drive/MyDrive/input/phim1.mp4`
- Output: `/content/drive/MyDrive/output/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 1. Cài ffmpeg
!apt-get update -qq
!apt-get install -y -qq ffmpeg

In [ ]:
# 2. Clone repo (xóa bản cũ nếu chạy lại notebook)
import os
if os.path.exists('/content/facefusionnnnnn'):
    !rm -rf /content/facefusionnnnnn

!git clone https://github.com/hunguet123/facefusionnnnnn.git
%cd /content/facefusionnnnnn

In [ ]:
# 3. Cài FaceFusion (CUDA)
!python install.py --onnxruntime cuda --skip-conda

In [ ]:
# 3.1 TensorRT libs
!pip install -q tensorrt==10.12.0.36

In [ ]:
# 3.2 LD_LIBRARY_PATH cho TensorRT (áp dụng cho cả shell lẫn Python)
import os, sys

site_packages = [p for p in sys.path if 'site-packages' in p or 'dist-packages' in p][0]
tensorrt_libs = f"{site_packages}/tensorrt_libs"
ld_path = os.environ.get('LD_LIBRARY_PATH', '') + f':{tensorrt_libs}'
os.environ['LD_LIBRARY_PATH'] = ld_path

# Persist cho lệnh !python ở cell sau
get_ipython().run_line_magic('env', f'LD_LIBRARY_PATH={ld_path}')
print('LD_LIBRARY_PATH OK:', tensorrt_libs)

In [ ]:
# 4. Patch facefusion.ini: Mac → Colab (giữ nguyên toàn bộ setting swap đã tối ưu trong repo)
with open('facefusion.ini', 'r', encoding='utf-8') as f:
    ini = f.read()

COLAB_PATCHES = {
    'output_video_encoder = h264_videotoolbox': 'output_video_encoder = h264_nvenc',
    'execution_providers = coreml': 'execution_providers = tensorrt cuda',
    'face_swapper_model = inswapper_128_fp16': 'face_swapper_model = inswapper_128',
    'system_memory_limit = 16': 'system_memory_limit = 0',
}

for old, new in COLAB_PATCHES.items():
    if old not in ini:
        print(f'⚠ Không tìm thấy (bỏ qua): {old}')
    else:
        ini = ini.replace(old, new)
        print(f'✓ {new}')

with open('facefusion.ini', 'w', encoding='utf-8') as f:
    f.write(ini)

print('\nConfig Colab đã sẵn sàng.')

In [ ]:
# 5. Bật Gradio public link
!find facefusion/uis/layouts -name "*.py" -exec sed -i 's/ui.launch(/ui.launch(share=True, /g' {} +

In [ ]:
# 6. Tạo thư mục trên Drive
!mkdir -p /content/drive/MyDrive/input/face
!mkdir -p /content/drive/MyDrive/output

In [ ]:
# 7. Kiểm tra ONNX providers (TensorRT + CUDA phải có)
!python -c "import onnxruntime as ort; print(ort.get_available_providers())"

In [ ]:
# 8. Chạy FaceFusion
# Dùng facefusion.ini từ repo (đã patch Colab ở bước 4)
# Chỉ truyền path + vài override Colab; không ghi đè setting swap trong ini

SOURCE = "/content/drive/MyDrive/input/face"
TARGET = "/content/drive/MyDrive/input/phim1.mp4"
OUTPUT = "/content/drive/MyDrive/output/"

!python facefusion.py run \
    --execution-providers tensorrt cuda \
    --execution-thread-count 4 \
    --video-memory-strategy tolerant \
    --system-memory-limit 0 \
    -s "{SOURCE}" \
    -t "{TARGET}" \
    -o "{OUTPUT}"